In [30]:
import numpy as np
import pandas as pd
import os
from sklearn.metrics import classification_report
import joblib
from pathlib import Path


In [31]:
MODELS_PATH = "/home/mariya/Data/pallas/models/dnam"
DATASETS_PATH = "/home/mariya/Data/pallas/datasets/dnam"

DATASETS_PATH = "/home/mariya/Data/pallas/mlomix_datasets"
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, "cal-and-test-dnam")
X_test_path = os.path.join(TEST_DATASET_PATH, "features.dnam.csv")
y_test_path = os.path.join(TEST_DATASET_PATH, "labels.csv")
random_seed = 42
rf_model_path = os.path.join(MODELS_PATH, "rf_model.pkl")
selected_features_path = os.path.join(MODELS_PATH, "selected_features.txt")
pipeline_path = os.path.join(MODELS_PATH, "pipeline.pkl")
hyperparams_path = os.path.join(MODELS_PATH, "hyperparams.pkl")

base_dir = Path.cwd()
selected_features_list_path = base_dir / "output" / "selected_features.4.txt"

In [32]:
# Read in selected features as list
with open(selected_features_list_path, "r") as f:
    selected_features = [line.strip() for line in f.readlines()]

In [33]:
X_test = pd.read_csv(X_test_path, index_col=0)
y_test = pd.read_csv(y_test_path, index_col=0)
X_test, y_test

(              cg00000029  cg00000109  cg00000165  cg00000236  cg00000289  \
 id                                                                         
 ALL_1252          0.8935      0.6426      0.1706      0.9034      0.8100   
 ALL_1281          0.7346      0.9177      0.1044      0.8865      0.6441   
 ALL_1290          0.8925      0.9237      0.8276      0.9143      0.7599   
 ALL_1362          0.8582      0.9206      0.0536      0.9246      0.8610   
 ALL_1378          0.6652      0.8986      0.8423      0.8923      0.7402   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2990      0.9410      0.9483      0.1263      0.9278      0.8406   
 TCGA-AB-2998      0.8964      0.9439      0.0988      0.5311      0.8669   
 TCGA-AB-2999      0.8241      0.9196      0.1252      0.4246      0.8446   
 TCGA-AB-3001      0.3904      0.9009      0.4026      0.5815      0.8237   
 TCGA-AB-3011      0.4893      0.9445      0.2474      0.5835      0.8439   

In [34]:
# Get just selected features
X_test = X_test[selected_features]
X_test.shape

(734, 1227)

In [35]:
# align the indices
X_test = X_test.loc[y_test.index]
X_test, y_test

(              cg07232063  cg15172966  cg21465180  cg13064897  cg14992144  \
 id                                                                         
 ALL_1252          0.7145      0.9328      0.1490      0.6430      0.0655   
 ALL_1281          0.9099      0.9247      0.0646      0.9075      0.3982   
 ALL_1290          0.9018      0.9310      0.1995      0.8942      0.1606   
 ALL_1362          0.9214      0.9300      0.0564      0.9425      0.1042   
 ALL_1378          0.8490      0.9265      0.1724      0.8962      0.1220   
 ...                  ...         ...         ...         ...         ...   
 TCGA-AB-2990      0.9227      0.9736      0.4678      0.9415      0.0704   
 TCGA-AB-2998      0.9585      0.9579      0.0861      0.9446      0.0435   
 TCGA-AB-2999      0.9494      0.9623      0.1059      0.8914      0.1045   
 TCGA-AB-3001      0.8653      0.8953      0.3869      0.7479      0.1859   
 TCGA-AB-3011      0.7480      0.9023      0.1709      0.6409      0.0774   

In [36]:
print("\ny_test class distribution:")
print(y_test.value_counts())


y_test class distribution:
class                   
AML, KMT2A-r                103
B-ALL, high hyperdiploid     98
B-ALL, ETV6::RUNX1           85
AML, RUNX1::RUNX1T1          74
AML, NPM1alt                 56
AML, CBFB::MYH11             54
Non-leukemic                 41
T-ALL                        32
AML, NUP98-r                 30
AML, CEBPAalt                23
B-ALL, PAX5alt               20
B-ALL, KMT2A-r               19
B-ALL, TCF3::PBX1            11
B-ALL, DUX4-r                10
AML, CBFA2T3::GLIS2          10
B-ALL, BCR::ABL1              8
AML, DEK::NUP214              8
AML, APL-PML::RARA            7
B-ALL, iAMP21                 7
AML, ETV6-r                   7
B-ALL, ZNF384-r               6
B-ALL, ETV6::RUNX1-like       6
B-ALL, BCR::ABL1-like         6
AML, KAT6A-r                  3
B-ALL, MEF2D-r                3
AML, PICALM::MLLT10           3
AML, FUS::ERG                 2
B-ALL, PAX5 P80R              2
Name: count, dtype: int64


In [37]:
y_test = y_test.iloc[:, 0].values # Convert to 1D numpy array
y_test

array(['B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, ETV6::RUNX1', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, ETV6::RUNX1',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'AML, KMT2A-r', 'AML, NPM1alt',
       'AML, KMT2A-r', 'AML, CEBPAalt', 'AML, CBFB::MYH11',
       'AML, CBFB::MYH11', 'AML, CEBPAalt', 'AML, RUNX1::RUNX1T1',
       'AML, CBFB::MYH11', 'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1',
       'AML, RUNX1::RUNX1T1', 'AML, CBFB::MYH11', 'AML, KMT2A-r',
       'AML, KMT2A-r', 'AML, RUNX1::RUNX1T1', 'AML, CEBPAalt',
       'AML, APL-PML::RARA', 'AML, CBFB::MYH11', 'AML, CBFB::MYH11',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, high hyperdiploid', 'B-ALL, PAX5alt',
       'B-ALL, high hyperdiploid', 'B-ALL, high hyperdiploid',
       'B-ALL, ETV6::RUNX1', 'B-ALL, PAX5alt', 'B-ALL, high hyperdiploid

In [39]:
# Load model
final_rf = joblib.load(rf_model_path)

# Load selected features from text file
selected_features_path = pd.read_csv(selected_features_list_path, header=None)[0].tolist()

# Use selected features for test set
y_pred_final = final_rf.predict(X_test)

# Print size of selected features
print(f"Number of selected features: {len(selected_features)}")

# Print shape of X_test_selected
print(f"Shape of X_test_selected: {X_test.shape}")

Number of selected features: 1227
Shape of X_test_selected: (734, 1227)


In [40]:
print("\nTest set classification report (final optimized model):")
print(classification_report(y_test, y_pred_final, digits=3))



Test set classification report (final optimized model):
                          precision    recall  f1-score   support

      AML, APL-PML::RARA      1.000     0.857     0.923         7
     AML, CBFA2T3::GLIS2      1.000     0.700     0.824        10
        AML, CBFB::MYH11      0.981     0.963     0.972        54
           AML, CEBPAalt      1.000     0.957     0.978        23
        AML, DEK::NUP214      1.000     0.875     0.933         8
             AML, ETV6-r      1.000     0.857     0.923         7
           AML, FUS::ERG      1.000     1.000     1.000         2
            AML, KAT6A-r      1.000     0.333     0.500         3
            AML, KMT2A-r      0.770     0.942     0.847       103
            AML, NPM1alt      0.960     0.857     0.906        56
            AML, NUP98-r      0.941     0.533     0.681        30
     AML, PICALM::MLLT10      0.500     1.000     0.667         3
     AML, RUNX1::RUNX1T1      0.986     0.986     0.986        74
        B-ALL, BCR

/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/mariya/miniconda3/envs/pallas/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [41]:
# ----------------------
# Extract top cpg sites from final model
# ----------------------
# Get feature importances from final RF
importances = final_rf.feature_importances_

# Rank genes by importance
gene_importances = pd.DataFrame({
    "Cpg Site": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 20 cpg sites driving classification (final model):")
print(gene_importances.head(20))


Top 20 cpg sites driving classification (final model):
       Cpg Site  Importance
81   cg06891858    0.009203
87   cg00015639    0.008664
1    cg15172966    0.008619
83   cg18638389    0.007173
86   cg10443049    0.006952
88   cg04646674    0.006265
4    cg14992144    0.005731
97   cg00587115    0.005491
94   cg08166720    0.005276
0    cg07232063    0.005108
93   cg18082788    0.005049
31   cg12452909    0.004870
10   cg21359950    0.004796
71   cg08616111    0.004728
5    cg25287071    0.004643
157  cg11367658    0.004568
68   cg13466180    0.004526
84   cg26129545    0.004493
8    cg16909799    0.004485
20   cg00765705    0.004400


In [42]:
# How many of the feature importances are zero?
num_zero_importances = np.sum(importances == 0)
print(f"\nNumber of features with zero importance: {num_zero_importances} out of {len(importances)}")

# Print lowest importance genes as well
print("\nBottom 20 genes by importance (final model):")
print(gene_importances.tail(20))


Number of features with zero importance: 0 out of 1227

Bottom 20 genes by importance (final model):
        Cpg Site  Importance
1042  cg26471982    0.000076
587   cg09326880    0.000075
1147  cg25587662    0.000072
772   cg27424447    0.000064
697   cg07531663    0.000063
688   cg07062885    0.000062
922   cg07997434    0.000057
1152  cg07085815    0.000056
823   cg18638931    0.000056
1200  cg15981092    0.000055
953   cg06697448    0.000052
973   cg02021919    0.000049
1136  cg10059959    0.000040
822   cg22572071    0.000036
1204  cg07737564    0.000036
940   cg04293152    0.000036
768   cg17734802    0.000035
668   cg19639021    0.000029
1008  cg03433404    0.000025
877   cg24039697    0.000017
